In [ ]:
# importing required libraries
import pandas as pd
import requests
import time
from bs4 import BeautifulSoup
from io import StringIO

In [ ]:
url = 'https://www.basketball-reference.com/leagues/NBA_2026_ratings.html'
data = requests.get(url)

soup = BeautifulSoup(data.text) # parsing the html with beautiful soup
table = soup.find_all('table', id='ratings')[0] # getting the ratings table

In [ ]:
links = table.find_all('a') # getting all the anchor tags/links in the table
links = [l.get("href") for l in links] # getting the urls of all the links

team_urls = [l for l in links if '/teams/' in l] # filtering to only get team urls
team_urls = [f"https://www.basketball-reference.com{l}" for l in team_urls] # formating to get the absolute urls for team urls

In [ ]:
all_teams = [] # To store the DataFrames of each team

for team_url in team_urls:
    team_abb = team_url.split("/")[-2] # getting the team abbreviation
    
    data = requests.get(team_url)
    soup = BeautifulSoup(data.text)
    stats = soup.find_all('table', id='per_game_stats')
    
    team_data = pd.read_html(StringIO(str(stats)))[0] # turning the html into a pandas DataFrame
    team_data.insert(3, "Team", team_abb)
    team_data = team_data.drop("Awards", axis=1)
    team_data = team_data.iloc[:, 1:] # Added this after making the csv file to account for an edit. Could cause an issue.
    all_teams.append(team_data)
    time.sleep(5) # delay each iteration to make sure we don't get blocked from scraping

teams_df = pd.concat(all_teams)

# renaming columns (Added this after making the csv file to account for an edit)
teams_df = teams_df.rename(columns={
    "FGA": "fg_avg",
    "FG%": "fg_pct",
    "3P": "threep",
    "3PA": "threep_avg",
    "3P%": "threep_pct",
    "2P": "twop",
    "2PA": "twop_avg",
    "2P%": "twop_pct",
    "eFG%": "efg_pct",
    "FTA": "ft_avg",
    "FT%": "ft_pct"
})
teams_df.columns = teams_df.columns.str.lower()

teams_df.to_csv("stats.csv")